In [1]:
# ─────────────────────────────────────────────
# CELL 1 — Dependencies
# ─────────────────────────────────────────────
import sys
!{sys.executable} -m pip install -q scikit-image lpips clean-fid opencv-python-headless
print("Dependencies ready")

Dependencies ready



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# ─────────────────────────────────────────────
# CELL 2 — Paths & Config
# ─────────────────────────────────────────────
from pathlib import Path
import random, json
import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# ── Synthetic Occlusion Dataset (từ notebook 01) ──
# Đây là dataset chính dùng cho evaluation (không thay đổi)
ROOT      = Path("/kaggle/input/datasets/dangvy1507/occlusion")
SYNTH_DIR = ROOT / "synthetic_occ"
GT_DIR    = SYNTH_DIR / "x_gt"
OCC_DIR   = SYNTH_DIR / "x_occ"
MASK_DIR  = SYNTH_DIR / "masks"
META_CSV  = SYNTH_DIR / "metadata_synthetic_occ.csv"

# ── Stanford Cars gốc (tham khảo - từ link bạn gửi)
# Không dùng trực tiếp ở đây, chỉ để kiểm tra
STANFORD_ROOT = Path("/kaggle/input/stanford-cars-dataset")
print(f"Stanford Cars root (tham khảo): {STANFORD_ROOT}")

# ── x_hat dirs từ notebook 02, 03, 04
XHAT_DIRS = {
    "A_SD_only"         : Path("/kaggle/working/baseline_sd2/x_hat"),
    "B_ControlNet_Canny": Path("/kaggle/working/controlnet_sd/x_hat"),
    "C_ControlNet_IP"   : Path("/kaggle/working/controlnet_ip_sd/x_hat"),
    "D_Text_Color"      : Path("/kaggle/working/controlnet_ip_sd/x_hat"),
}

# ── PER_IMAGE_SRC với FALLBACK tên file cũ từ notebook 02 & 03
PER_IMAGE_SRC = {
    "A_SD_only"         : Path("/kaggle/working/baseline_sd2/reports/baseline_metrics_per_image.csv"),
    "B_ControlNet_Canny": Path("/kaggle/working/controlnet_sd/reports/controlnet_metrics_per_image.csv"),
}

SUMMARY_SRC = {
    "A_SD_only"         : Path("/kaggle/working/baseline_sd2/reports/baseline_summary.csv"),
    "B_ControlNet_Canny": Path("/kaggle/working/controlnet_sd/reports/controlnet_summary.csv"),
}

# ── Notebook 04 dynamic (C or D)
IP_REPORT_DIR = Path("/kaggle/working/controlnet_ip_sd/reports")

# ── Output
EVAL_DIR = Path("/kaggle/working/eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_ORDER = ["A_SD_only", "B_ControlNet_Canny", "C_ControlNet_IP", "D_Text_Color"]

EVAL_CSV_NAMES = {
    "A_SD_only"         : "metrics_per_image_A_SD_only.csv",
    "B_ControlNet_Canny": "metrics_per_image_B_ControlNet_Canny.csv",
    "C_ControlNet_IP"   : "metrics_per_image_C_ControlNet_IP.csv",
    "D_Text_Color"      : "metrics_per_image_D_Text_Color.csv",
}

# Kiểm tra
print("\nKiểm tra synthetic dataset:")
print(f"  META_CSV exists : {META_CSV.exists()}")
print(f"  GT_DIR images   : {len(list(GT_DIR.glob('*.png'))) if GT_DIR.exists() else 0}")

print("\nKiểm tra x_hat dirs:")
for cfg, d in XHAT_DIRS.items():
    n = len(list(d.glob("*.png"))) if d.exists() else 0
    status = "✓" if n > 0 else "⚠"
    print(f"  {cfg:<25} {status} — {n} images")

Device: cpu
Stanford Cars root (tham khảo): \kaggle\input\stanford-cars-dataset

Kiểm tra synthetic dataset:
  META_CSV exists : False
  GT_DIR images   : 0

Kiểm tra x_hat dirs:
  A_SD_only                 ⚠ — 0 images
  B_ControlNet_Canny        ⚠ — 0 images
  C_ControlNet_IP           ⚠ — 0 images
  D_Text_Color              ⚠ — 0 images


In [ ]:
# ─────────────────────────────────────────────
# CELL 3 — Auto-detect config C/D từ notebook 04
# ─────────────────────────────────────────────

nb04_label   = None   # 'C_SD_CN_IP' hoặc 'D_SD_CN_TextColor'
nb04_abl_key = None   # 'C_ControlNet_IP' hoặc 'D_Text_Color'

for candidate in ["C_SD_CN_IP", "D_SD_CN_TextColor"]:
    pi  = IP_REPORT_DIR / f"{candidate}_metrics_per_image.csv"
    sm  = IP_REPORT_DIR / f"{candidate}_summary.csv"
    if pi.exists():
        nb04_label   = candidate
        nb04_abl_key = "C_ControlNet_IP" if "C_SD_CN_IP" == candidate else "D_Text_Color"
        PER_IMAGE_SRC[nb04_abl_key] = pi
        SUMMARY_SRC[nb04_abl_key]   = sm
        print(f"Detected notebook 04 config : {candidate}")
        print(f"  Ablation key             : {nb04_abl_key}")
        print(f"  Per-image CSV            : {pi}")
        print(f"  Summary CSV              : {sm}")
        break

if nb04_label is None:
    print("⚠  Không tìm thấy kết quả notebook 04")
    print("   Nếu bạn đã chạy 04, kiểm tra lại đường dẫn trong IP_REPORT_DIR")

# Danh sách config cần xử lý
ACTIVE = ["A_SD_only", "B_ControlNet_Canny"]
if nb04_abl_key:
    ACTIVE.append(nb04_abl_key)
print(f"\nConfigs sẽ xử lý: {ACTIVE}")

⚠  Không tìm thấy kết quả notebook 04
   Nếu bạn đã chạy 04, kiểm tra lại đường dẫn trong IP_REPORT_DIR

Configs sẽ xử lý: ['A_SD_only', 'B_ControlNet_Canny']


In [ ]:
# ─────────────────────────────────────────────
# CELL 4 — Load metadata + LPIPS model
# ─────────────────────────────────────────────
import lpips
import cv2
from PIL import Image
from skimage.metrics import structural_similarity as ssim_fn

meta = pd.read_csv(META_CSV)
meta.columns = meta.columns.str.strip().str.lower()
print(f"Metadata: {len(meta)} rows | columns: {meta.columns.tolist()}")

# Load LPIPS một lần — chỉ dùng nếu phải tính lại
lpips_model = lpips.LPIPS(net="alex").to(DEVICE)
lpips_model.eval()
print(f"LPIPS AlexNet loaded on {DEVICE}")


# ── Metric functions (giữ đúng convention của 02/03/04) ──
def masked_ssim(gt_rgb: np.ndarray, pred_rgb: np.ndarray, mask01: np.ndarray) -> float:
    gt_f = gt_rgb.astype(np.float32) / 255.0
    pr_f = pred_rgb.astype(np.float32) / 255.0
    _, ssim_map = ssim_fn(gt_f, pr_f, channel_axis=2, data_range=1.0, full=True)
    m = mask01.astype(bool)
    return float(ssim_map[m].mean()) if m.sum() > 0 else float(ssim_map.mean())


def masked_lpips(gt_rgb: np.ndarray, pred_rgb: np.ndarray, mask01: np.ndarray) -> float:
    m    = mask01.astype(np.float32)[..., None]
    gt_m = (gt_rgb.astype(np.float32) * m).astype(np.uint8)
    pr_m = (pred_rgb.astype(np.float32) * m).astype(np.uint8)
    gt_t = torch.from_numpy(gt_m).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1.0
    pr_t = torch.from_numpy(pr_m).permute(2,0,1).unsqueeze(0).float() / 127.5 - 1.0
    with torch.no_grad():
        return float(lpips_model(gt_t.to(DEVICE), pr_t.to(DEVICE)).item())


def load_triplet(stem: str, xhat_dir: Path):
    """Load (gt_rgb, pred_rgb, mask01). Return None nếu thiếu file."""
    gt_cands = list(GT_DIR.glob(f"{stem}.*"))
    if not gt_cands:
        return None
    gt = cv2.cvtColor(cv2.resize(cv2.imread(str(gt_cands[0])), (512,512)), cv2.COLOR_BGR2RGB)

    pred_path = xhat_dir / f"{stem}.png"
    if not pred_path.exists():
        return None
    pred = cv2.cvtColor(cv2.resize(cv2.imread(str(pred_path)), (512,512)), cv2.COLOR_BGR2RGB)

    # 02/03/04 dùng: MASK_DIR / f"{stem}.png"
    mask_path = MASK_DIR / f"{stem}.png"
    if not mask_path.exists():
        mask_cands = list(MASK_DIR.glob(f"{stem}.*"))
        if not mask_cands:
            return None
        mask_path = mask_cands[0]
    mask01 = (cv2.resize(cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE), (512,512)) > 127).astype(np.uint8)

    return gt, pred, mask01


print("Metric functions defined.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 5 — Xử lý per-image metrics từng config
# ─────────────────────────────────────────────
from tqdm.auto import tqdm

REQUIRED_COLS = {"stem", "occlusion_ratio", "ssim", "lpips"}
ablation_rows = []

for cfg in ACTIVE:
    out_csv  = EVAL_DIR / EVAL_CSV_NAMES[cfg]
    src_csv  = PER_IMAGE_SRC.get(cfg)
    xhat_dir = XHAT_DIRS.get(cfg)
    df_cfg   = None

    print(f"\n══ {cfg} ══")

    # ── Thử reuse CSV từ notebook 02/03/04 ──────────────────
    if src_csv and src_csv.exists():
        df_tmp = pd.read_csv(src_csv)
        df_tmp.columns = df_tmp.columns.str.strip().str.lower()
        if REQUIRED_COLS.issubset(set(df_tmp.columns)):
            df_cfg = df_tmp[list(REQUIRED_COLS)].copy()
            df_cfg["config"] = cfg
            print(f"  ✓ Reused: {src_csv.name}  ({len(df_cfg)} rows)")
        else:
            missing = REQUIRED_COLS - set(df_tmp.columns)
            print(f"  ⚠  CSV thiếu cột {missing} → tính lại từ x_hat")
    else:
        print(f"  CSV không tìm thấy → tính lại từ x_hat")

    # ── Tính lại từ x_hat nếu cần ──────────────────────────
    if df_cfg is None:
        if xhat_dir is None or not xhat_dir.exists():
            print(f"  ⚠  x_hat dir không tồn tại: {xhat_dir}")
            print(f"     → Skip. Chạy notebook 02/03/04 trước.")
            continue

        stems = sorted([p.stem for p in xhat_dir.glob("*.png")])
        if not stems:
            print(f"  ⚠  0 images trong {xhat_dir} → Skip")
            continue

        print(f"  Tính lại: {len(stems)} images...")
        rows = []
        skipped = 0
        for stem in tqdm(stems, desc=cfg, leave=False):
            result = load_triplet(stem, xhat_dir)
            if result is None:
                skipped += 1
                continue
            gt, pred, mask01 = result
            meta_row  = meta[meta["stem"] == stem]
            occ_ratio = float(meta_row["occlusion_ratio"].values[0]) \
                        if len(meta_row) > 0 else np.nan
            rows.append({
                "stem"            : stem,
                "occlusion_ratio" : occ_ratio,
                "ssim"            : round(masked_ssim(gt, pred, mask01), 6),
                "lpips"           : round(masked_lpips(gt, pred, mask01), 6),
                "config"          : cfg,
            })
        if skipped:
            print(f"  ⚠  Skipped {skipped} stems (missing gt/pred/mask)")
        if not rows:
            print(f"  ⚠  Không tính được metrics → Skip")
            continue
        df_cfg = pd.DataFrame(rows)
        print(f"  Computed: {len(df_cfg)} rows")

    # ── Lưu ra eval/ ────────────────────────────────────────
    df_cfg.to_csv(out_csv, index=False)
    print(f"  Saved  → {out_csv.name}")
    print(f"  SSIM   mean={df_cfg['ssim'].mean():.4f}  std={df_cfg['ssim'].std():.4f}")
    print(f"  LPIPS  mean={df_cfg['lpips'].mean():.4f}  std={df_cfg['lpips'].std():.4f}")

    ablation_rows.append({
        "config"    : cfg,
        "ssim_mean" : round(df_cfg["ssim"].mean(),  6),
        "ssim_std"  : round(df_cfg["ssim"].std(),   6),
        "lpips_mean": round(df_cfg["lpips"].mean(), 6),
        "lpips_std" : round(df_cfg["lpips"].std(),  6),
        "n_images"  : len(df_cfg),
        "fid"       : np.nan,  # điền ở cell sau
    })

print(f"\n✓ Per-image metrics done — {len(ablation_rows)} configs")

In [ ]:
# ─────────────────────────────────────────────
# CELL 6 — Lấy FID + metadata từ summary CSVs
# ─────────────────────────────────────────────
EXTRA_COLS = ["model", "scheduler", "steps", "guidance",
              "cn_scale", "ip_scale", "prompt", "negative_prompt"]

for row in ablation_rows:
    cfg      = row["config"]
    sum_path = SUMMARY_SRC.get(cfg)

    if sum_path and sum_path.exists():
        df_sum = pd.read_csv(sum_path)
        df_sum.columns = df_sum.columns.str.strip().str.lower()

        # FID
        if "fid" in df_sum.columns:
            row["fid"] = round(float(df_sum["fid"].values[0]), 4)
            print(f"  {cfg:<25} FID={row['fid']:.2f}  (from {sum_path.name})")
        else:
            print(f"  {cfg:<25} ⚠  không có cột 'fid' trong {sum_path.name}")

        # Metadata thêm
        for col in EXTRA_COLS:
            if col in df_sum.columns:
                row[col] = df_sum[col].values[0]
    else:
        print(f"  {cfg:<25} ⚠  summary CSV không tìm thấy: {sum_path}")

# Tính lại FID nếu còn thiếu
missing_fid = [r for r in ablation_rows if pd.isna(r["fid"])]
if missing_fid:
    print(f"\n{len(missing_fid)} config thiếu FID — tính lại với clean-fid...")
    from cleanfid import fid as cleanfid

    # GT dir: dùng eval_gt_200 của config A (đã có từ notebook 02)
    FID_GT = Path("/kaggle/working/baseline_sd2/eval_gt_200")
    if not FID_GT.exists() or len(list(FID_GT.glob("*.png"))) == 0:
        # Fallback: build từ GT_DIR theo stems của config A
        FID_GT = EVAL_DIR / "fid_gt_tmp"
        FID_GT.mkdir(exist_ok=True)
        a_csv_path = EVAL_DIR / EVAL_CSV_NAMES["A_SD_only"]
        if a_csv_path.exists():
            a_stems = pd.read_csv(a_csv_path)["stem"].tolist()
            for stem in tqdm(a_stems, desc="Building FID GT", leave=False):
                cands = list(GT_DIR.glob(f"{stem}.*"))
                if cands and not (FID_GT / f"{stem}.png").exists():
                    Image.open(cands[0]).convert("RGB").resize((512,512)).save(FID_GT / f"{stem}.png")
        print(f"  FID GT dir built: {len(list(FID_GT.glob('*.png')))} images")

    for row in missing_fid:
        cfg      = row["config"]
        xhat_dir = XHAT_DIRS.get(cfg)
        if not (xhat_dir and xhat_dir.exists()):
            continue
        print(f"  Computing FID for {cfg}...", end=" ", flush=True)
        try:
            fid_score = cleanfid.compute_fid(
                str(FID_GT), str(xhat_dir),
                device=torch.device(DEVICE),
                num_workers=2, verbose=False,
            )
            row["fid"] = round(float(fid_score), 4)
            print(f"FID={fid_score:.2f}")
        except Exception as e:
            print(f"FAILED: {e}")

print("\n✓ FID done.")

In [ ]:
# ─────────────────────────────────────────────
# CELL 7 — Xuất ablation_rows.csv + run_metadata.json
# ─────────────────────────────────────────────

if not ablation_rows:
    raise RuntimeError(
        "ablation_rows trống!\n"
        "Kiểm tra: (1) notebook 02/03/04 đã chạy, "
        "(2) đường dẫn XHAT_DIRS đúng, "
        "(3) CSV từ 02/03/04 có cột stem/occlusion_ratio/ssim/lpips."
    )

abl_df = pd.DataFrame(ablation_rows)
abl_df["config"] = pd.Categorical(
    abl_df["config"], categories=CONFIG_ORDER, ordered=True
)
abl_df = abl_df.sort_values("config").reset_index(drop=True)

abl_path = EVAL_DIR / "ablation_rows.csv"
abl_df.to_csv(abl_path, index=False)
print(f"Saved: {abl_path}")

print("\n══════════════════════════════════════════════════════")
print("  UNIFIED METRICS SUMMARY")
print("══════════════════════════════════════════════════════")
show_cols = [c for c in ["config","ssim_mean","ssim_std","lpips_mean","lpips_std","fid","n_images"]
             if c in abl_df.columns]
print(abl_df[show_cols].to_string(index=False))
print("══════════════════════════════════════════════════════")

run_meta = {
    "seed"             : SEED,
    "device"           : DEVICE,
    "lpips_net"        : "alex",
    "img_size"         : 512,
    "configs_evaluated": [r["config"] for r in ablation_rows],
    "eval_dir"         : str(EVAL_DIR),
    "per_image_csvs"   : {
        cfg: str(EVAL_DIR / EVAL_CSV_NAMES[cfg])
        for cfg in ACTIVE if cfg in EVAL_CSV_NAMES
    },
}
(EVAL_DIR / "run_metadata.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")
print("Saved: run_metadata.json")

print("\n── Notebook 05 complete ──")
print("Tiếp theo: chạy 06_ablation-analysis.ipynb (không cần GPU)")